<a href="https://colab.research.google.com/github/NickolasGeyfman/dl_sar_despeckling/blob/main/dl_sar_despeckling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sections:

	1.	Introduction: Speckle noise, why it’s multiplicative, how classical filters work.

	2.	Classical Experiments: Code + results (images & metrics).

	3.	Deep Learning: Dataset setup, model architecture, training process.

	4.	Comparison: Evaluate performance side by side (tables, graphs, example images).


Learning Approaches:

	1.	Noise2Noise
	•	You train using pairs of noisy images of the same scene (or nearly the same scene) at two different times/angles.
	•	The idea: each noisy image is slightly different noise realization, so training on pairs can lead the model to learn the underlying “clean” signal.
	2.	Noise2Void / Speckle2Void
	•	Uses only single noisy images—no second image or “clean” reference required.
	•	Exploits a “blind-spot” or “masking” technique so the network cannot cheat by seeing the same pixel during training.
	•	This forces the model to predict each pixel from its neighbors and effectively learn to remove noise.
	3.	SAR2SAR
	•	Specifically built for SAR speckle. Similar principle: your “labels” are just other noisy images or sub-pixel shifts.
	•	Results can be surprisingly good, despite not having a perfect clean reference.

These self‐supervised methods are exactly how advanced denoising models are trained without needing a true “clean” image.

In [ ]:
!pip install rasterio
!pip install gdal
!pip install torch torchvision


In [ ]:
# 1) Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Dataset, Subset
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import os
import random
from scipy.ndimage import uniform_filter

# 2) Hyperparameters & Config
BATCH_SIZE = 4
LR = 1e-4
BETAS = (0.9, 0.95)
EPS = 1e-7
GRAD_CLIP_MAX_NORM = 0.1
NUM_EPOCHS = 30
PATIENCE = 5
MASK_RATIO = 0.1
PRINT_INTERVAL = 50
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


First let's demonstrate traditional techniques:

We will use the Lee, Frost, and Kuan speckle filters.

In [ ]:
from scipy.ndimage import uniform_filter

def lee_filter(img, kernel_size=3, var_noise=0.25):
    """
    Basic additive Lee Filter:
    - local mean + weight*(pixel - local mean)
    - weight = var_local / (var_local + var_noise)
    """
    # 1) Local mean of img
    mean_window = uniform_filter(img, kernel_size)
    # 2) Local mean of squares
    mean_sqr_window = uniform_filter(img**2, kernel_size)
    # 3) Local variance
    var_window = mean_sqr_window - mean_window**2

    # 4) Weight
    weight = var_window / (var_window + var_noise)
    # 5) Output
    img_filtered = mean_window + weight*(img - mean_window)
    return img_filtered

def frost_filter(img, kernel_size=3, var_noise=0.25):
    """
    Basic Frost Filter:
    - Weight matrix w(i,j) = exp(-k * local_variance(i,j) * distance)
    - For simplicity, we’ll use a uniform "distance" approach or just local variance weighting.
    NOTE: This minimal example may not fully replicate the directional aspect of Frost.
    """
    # We'll do a simplified approach that uses local mean/variance,
    # weighting by an exponential function of variance.

    mean_window = uniform_filter(img, kernel_size)
    mean_sqr_window = uniform_filter(img**2, kernel_size)
    var_window = mean_sqr_window - mean_window**2

    # Freedman simplification: w = exp(-k * var_window) with some constant k
    # Adjust k based on var_noise for demonstration
    k = 1.0 / (var_noise+1e-6)
    weights = np.exp(-k * var_window)

    # We'll do a simple normalized weighting:
    # Weighted image = weights*img + (1 - weights)*mean
    # This is a very simplified approach to demonstrate an "exponential" weighting idea

    img_filtered = weights*img + (1 - weights)*mean_window
    return img_filtered

def kuan_filter(img, kernel_size=3, var_noise=0.25):
    """
    Basic Kuan Filter:
    - Similar local-statistics approach:
      local mean + alpha*(pixel - local mean)
    - alpha = (var_local - var_noise) / var_local
    - For multiplicative noise, can do log transforms, etc. Here we do a simple additive version.
    """
    mean_window = uniform_filter(img, kernel_size)
    mean_sqr_window = uniform_filter(img**2, kernel_size)
    var_window = mean_sqr_window - mean_window**2

    # In Kuan's approach: alpha = max(0, 1 - var_noise / var_local)
    # We'll do a basic version:
    # But note, many references do alpha = (var_local - var_noise) / var_local
    # We'll clamp alpha to [0,1]
    alpha = (var_window - var_noise) / (var_window + 1e-6)
    alpha = np.clip(alpha, 0, 1)

    img_filtered = mean_window + alpha*(img - mean_window)
    return img_filtered

In [ ]:
def apply_filter_in_log_domain(sub_img, filter_func, **kwargs):
    # 1) log transform
    log_img = np.log1p(sub_img)  # log(1 + x)

    # 2) apply the existing additive-based filter
    log_filtered = filter_func(log_img, **kwargs)

    # 3) exponentiate (minus 1) to return to linear scale
    filtered_lin = np.expm1(log_filtered)
    return filtered_lin
def show_with_clipping(ax, arr, title="", clip_low=2, clip_high=98, cmap='gray'):
    """
    Displays arr with intensity clipped between the clip_low-th and clip_high-th percentiles.
    """
    vmin = np.percentile(arr, clip_low)
    vmax = np.percentile(arr, clip_high)
    ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')


In [ ]:
def show_subregion(tif_path):
    with rasterio.open(tif_path) as ds:
        img = ds.read(1).astype(np.float32)
    H, W = img.shape
    print("Loaded image shape:", (H, W))

    SUB_SIZE = 512

    # Hardcoded row, col
    row = 8232
    col = 15410
    print(f"Using subregion at row={row}, col={col}")

    sub_img = img[row:row+SUB_SIZE, col:col+SUB_SIZE]

    # Apply each filter in log domain
    lee_res   = apply_filter_in_log_domain(sub_img, lee_filter,   kernel_size=5, var_noise=0.25)
    frost_res = apply_filter_in_log_domain(sub_img, frost_filter, kernel_size=5, var_noise=0.25)
    kuan_res  = apply_filter_in_log_domain(sub_img, kuan_filter,  kernel_size=5, var_noise=0.25)

    fig, axes = plt.subplots(1, 4, figsize=(18,6))

    show_with_clipping(axes[0], sub_img,     title="Original")
    show_with_clipping(axes[1], lee_res,     title="Lee")
    show_with_clipping(axes[2], frost_res,   title="Frost")
    show_with_clipping(axes[3], kuan_res,    title="Kuan")

    plt.tight_layout()
    plt.show()

In [ ]:
tif_path = "/content/drive/MyDrive/sentineldata/tiff-files/S1A_IW_GRDH_1SDV_20250118T050836_20250118T050901_057494_071483_42ED_COG.SAFE_measurement/s1a-iw-grd-vh-20250118t050836-20250118t050901-057494-071483-002-cog.tiff"
show_subregion(tif_path)

# Data Preprocessing Step

We prepare VV and VH Sentinel‑1 data by reading each polarization into separate arrays, applying a log1p transform to stabilize amplitudes and approximate multiplicative noise. We then stack VV and VH into a 2-channel format, letting the model exploit distinct scattering behaviors across polarizations. To handle large images, each 2D array is sliced into manageable 256×256 patches, and all patches are consolidated into a single dataset for uniform training of both U‑Net and transformer models. This multi-polarization approach often yields richer feature representation and improved speckle denoising compared to single-channel data.

In [ ]:
SOURCE_DIR = "/content/drive/MyDrive/tiff-files/sentineldata"
OUT_DIR = "/content/drive/MyDrive/sentineldata/patches_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

PATCH_SIZE = 256
STRIDE = 256
APPLY_LOG = True

def extract_patches(arr, patch_size=256, stride=256):
    C, H, W = arr.shape
    patches_list = []
    for row in range(0, H - patch_size + 1, stride):
        for col in range(0, W - patch_size + 1, stride):
            patch = arr[:, row:row+patch_size, col:col+patch_size]
            patches_list.append(patch)
    return patches_list

def process_folder(folder_path, folder_name):
    """Loads vv/vh from folder_path, extracts patches, saves to .npy, returns number of patches."""
    vv_path, vh_path = None, None
    for fname in os.listdir(folder_path):
        fname_lower = fname.lower()
        if fname_lower.endswith(".tiff") and "vv" in fname_lower:
            vv_path = os.path.join(folder_path, fname)
        elif fname_lower.endswith(".tiff") and "vh" in fname_lower:
            vh_path = os.path.join(folder_path, fname)

    if not vv_path or not vh_path:
        print(f"Skipping {folder_name}: no vv/vh tiffs found.")
        return 0

    with rasterio.open(vv_path) as ds_vv:
        vv_arr = ds_vv.read(1)
    with rasterio.open(vh_path) as ds_vh:
        vh_arr = ds_vh.read(1)

    if APPLY_LOG:
        vv_arr = np.log1p(vv_arr)
        vh_arr = np.log1p(vh_arr)

    stack_arr = np.stack([vv_arr, vh_arr], axis=0).astype(np.float32)

    folder_patches = extract_patches(stack_arr, PATCH_SIZE, STRIDE)
    folder_patches = np.array(folder_patches, dtype=np.float32)

    out_file = os.path.join(OUT_DIR, f"{folder_name}_patches.npy")
    np.save(out_file, folder_patches)
    print(f"{folder_name}: {len(folder_patches)} patches → {out_file}")

    # Free memory
    del stack_arr
    del vv_arr
    del vh_arr

    return len(folder_patches)

# Main loop
total_patches = 0
for folder_name in os.listdir(SOURCE_DIR):
    folder_path = os.path.join(SOURCE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    n = process_folder(folder_path, folder_name)
    total_patches += n

print(f"Done. Total patches across all folders: {total_patches}")

Now, I'm going to set up my Training/Validation/Test sets:

In [ ]:

# -------------------------------------------------------------------
# 1) Configure file paths and set random seed for reproducibility
# -------------------------------------------------------------------
npy_files = [
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T050836_20250118T050901_057494_071483_42ED_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T050901_20250118T050926_057494_071483_82EF_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T050926_20250118T050951_057494_071483_DF83_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T050951_20250118T051016_057494_071483_32FB_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051016_20250118T051041_057494_071483_B3B0_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051041_20250118T051106_057494_071483_302C_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051106_20250118T051131_057494_071483_9504_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051131_20250118T051156_057494_071483_E230_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051156_20250118T051221_057494_071483_914F_COG.SAFE_measurement_patches.npy",
    "/content/drive/MyDrive/sentineldata/patches_chunks/S1A_IW_GRDH_1SDV_20250118T051221_20250118T051246_057494_071483_9271_COG.SAFE_measurement_patches.npy",


]
train_ratio = 0.8
val_ratio   = 0.1
test_ratio  = 0.1

random_seed = 42
random.seed(random_seed)

# -------------------------------------------------------------------
# 2) Build a global index of all patches across all files
# -------------------------------------------------------------------
all_patches = []  # will store tuples of (file_path, patch_index)

for fpath in npy_files:
    # We open in read-only mode to avoid loading everything into memory
    data = np.load(fpath, mmap_mode='r')
    num_patches = data.shape[0]  # first dimension = number of patches
    for i in range(num_patches):
        all_patches.append((fpath, i))
    del data

print(f"Total patches across all files: {len(all_patches)}")

# -------------------------------------------------------------------
# 3) Shuffle and split by train/val/test ratios
# -------------------------------------------------------------------
random.shuffle(all_patches)

train_end = int(len(all_patches) * train_ratio)
val_end   = train_end + int(len(all_patches) * val_ratio)

train_patches = all_patches[:train_end]
val_patches   = all_patches[train_end:val_end]
test_patches  = all_patches[val_end:]

print(f"Train patches: {len(train_patches)}")
print(f"Val patches:   {len(val_patches)}")
print(f"Test patches:  {len(test_patches)}")

# -------------------------------------------------------------------
# 4) Save the splits in text/CSV or keep them in Python
# -------------------------------------------------------------------
# For example, we can save them as text files listing (fpath, idx).

def save_split(list_of_tuples, out_path):
    with open(out_path, 'w') as f:
        for fpath, idx in list_of_tuples:
            f.write(f"{fpath},{idx}\n")

os.makedirs("splits", exist_ok=True)
save_split(train_patches, "splits/train_split.txt")
save_split(val_patches,   "splits/val_split.txt")
save_split(test_patches,  "splits/test_split.txt")

print("Splits saved as text files in the 'splits' directory.")

#SarPatchDataset Class:

This dataset class is designed for loading SAR patch data with optional transforms (e.g., log transform for speckle noise). It does the following:

    1.  Reads a split file listing (file_path, patch_index) for each patch.
    2.  Loads the specified patch from a .npy file using np.load with mmap_mode='r'.
    3.  Converts the loaded patch to a torch.Tensor (float).
    4.  Treats inputs == targets (useful for self-supervised Noise2Void).
    5.  Optionally applies a transform function (like log_transform).

This approach ensures each patch is readily prepared for a Noise2Void-style training loop.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset

class SarPatchDataset(Dataset):
    def __init__(self, split_file, transform=None):
        """
        Args:
          split_file: a text file with lines like '/path/to/file.npy, patch_index'
          transform: a callable function that takes (inputs, targets)
                     and returns (transformed_inputs, transformed_targets)
        """
        self.samples = []
        with open(split_file, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                filepath, idx_str = line.split(',')
                self.samples.append((filepath, int(idx_str)))

        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, patch_idx = self.samples[idx]
        data = np.load(filepath, mmap_mode='r')
        patch = data[patch_idx]
        patch = np.copy(patch)  # make writable

        # Convert to torch Tensor
        patch_tensor = torch.from_numpy(patch).float()

        # For Noise2Void => inputs == targets
        inputs = patch_tensor
        targets = patch_tensor

        # If a transform is provided, apply it
        if self.transform:
            inputs, targets = self.transform(inputs, targets)

        return inputs, targets


#Log Transform & DataLoader Setup:

This snippet defines a log_transform function and then creates train/val/test datasets (with an optional log transform). Finally, it wraps those datasets in DataLoaders:

    1.  log_transform:
        - Clamps negative values to 0.0,
        - Applies torch.log1p to both inputs and targets,
        - Returns them in log domain.

    2.  SarPatchDataset with transform=log_transform:
        - This applies the log transform to each patch (inputs, targets) upon loading.

    3.  DataLoader instantiation:
        - Creates train_loader, val_loader, test_loader with batch_size=128 and given num_workers.
        - shuffle=True for train, False for val/test.

In [ ]:
import torch

def log_transform(inputs, targets):
    """
    Clamps to avoid negatives, then applies log1p to both inputs and targets.
    """
    inputs = torch.clamp_min(inputs, 0.0)
    targets = torch.clamp_min(targets, 0.0)
    inputs = torch.log1p(inputs)
    targets = torch.log1p(targets)
    return inputs, targets

# Now create the datasets with transform=log_transform
train_dataset = SarPatchDataset("splits/train_split.txt", transform=log_transform)
val_dataset   = SarPatchDataset("splits/val_split.txt",   transform=log_transform)
test_dataset  = SarPatchDataset("splits/test_split.txt",  transform=log_transform)

# Keep the DataLoader parameters the same
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=4)

#UNet Architecture:

This U-Net model is designed for image-to-image tasks such as denoising. It typically consists of:

    1.  Encoder Blocks (contracting path)
        - Each block does two convolutions + ReLU, then a downsampling (MaxPool2d).
        - Gradually reduces spatial dimensions while increasing feature channels.

    2.  Bottleneck
        - The bottom (most compressed) level, often just two conv layers, no further pooling.

    3.  Decoder Blocks (expanding path)
        - Each up-convolution doubles spatial dimensions, then concatenates skip features from the encoder.
        - Restores original resolution, combining fine-grained details from earlier layers with high-level features.

    4.  Final 1x1 Convolution
        - Converts the feature maps to the desired number of output channels.

In a Noise2Void setting, the U-Net predicts masked pixels from context. Whether you train in log-space or linear space, this architecture effectively handles local and global structures in SAR patches for speckle denoising.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    """
    A 4-level (deep) U-Net that maintains shapes from 256 -> 16 in the bottleneck.
    Perfect for tasks on 256×256 inputs (or higher/lower powers of 2).

    Args:
        in_channels (int):  Number of input channels. For SAR dual-pol, use 2.
        out_channels (int): Number of output channels. For denoising, often same as in_channels.
        init_features (int): Base number of feature maps (default=64).
                             The deeper layers multiply this factor.
    """
    def __init__(self, in_channels=1, out_channels=1, init_features=64):
        super(UNet, self).__init__()

        features = init_features

        # ----------------- ENCODER -------------------
        # Each "down" block has (double conv + optional norm + pool)
        self.enc1 = UNet._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc2 = UNet._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc3 = UNet._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc4 = UNet._block(features * 4, features * 8, name="enc4")
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # ----------------- BOTTLENECK -----------------
        self.bottleneck = UNet._block(features * 8, features * 16, name="bottleneck")

        # ----------------- DECODER --------------------
        # Each "up" block: upsample (ConvTranspose2D) + skip connection + double conv
        self.up4 = nn.ConvTranspose2d(
            features * 16, features * 8, kernel_size=2, stride=2
        )
        self.dec4 = UNet._block(features * 16, features * 8, name="dec4")

        self.up3 = nn.ConvTranspose2d(
            features * 8, features * 4, kernel_size=2, stride=2
        )
        self.dec3 = UNet._block(features * 8, features * 4, name="dec3")

        self.up2 = nn.ConvTranspose2d(
            features * 4, features * 2, kernel_size=2, stride=2
        )
        self.dec2 = UNet._block(features * 4, features * 2, name="dec2")

        self.up1 = nn.ConvTranspose2d(
            features * 2, features, kernel_size=2, stride=2
        )
        self.dec1 = UNet._block(features * 2, features, name="dec1")

        # ----------------- HEAD --------------------
        self.conv_out = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )

    def forward(self, x):
        # ----------------- ENCODER PASS -----------------
        e1 = self.enc1(x)    # => shape: [N, 64, 256, 256]
        p1 = self.pool1(e1)  # => shape: [N, 64, 128, 128]

        e2 = self.enc2(p1)   # => [N, 128, 128, 128]
        p2 = self.pool2(e2)  # => [N, 128, 64, 64]

        e3 = self.enc3(p2)   # => [N, 256, 64, 64]
        p3 = self.pool3(e3)  # => [N, 256, 32, 32]

        e4 = self.enc4(p3)   # => [N, 512, 32, 32]
        p4 = self.pool4(e4)  # => [N, 512, 16, 16]

        # ----------------- BOTTLENECK ------------------
        b  = self.bottleneck(p4)  # => [N, 1024, 16, 16]

        # ----------------- DECODER PASS ----------------
        # 1) up + concat + conv
        up_4 = self.up4(b)                        # => [N, 512, 32, 32]
        cat_4 = torch.cat((up_4, e4), dim=1)      # => [N, 1024, 32, 32]
        d4 = self.dec4(cat_4)                     # => [N, 512, 32, 32]

        up_3 = self.up3(d4)                       # => [N, 256, 64, 64]
        cat_3 = torch.cat((up_3, e3), dim=1)      # => [N, 512, 64, 64]
        d3 = self.dec3(cat_3)                     # => [N, 256, 64, 64]

        up_2 = self.up2(d3)                       # => [N, 128, 128, 128]
        cat_2 = torch.cat((up_2, e2), dim=1)      # => [N, 256, 128, 128]
        d2 = self.dec2(cat_2)                     # => [N, 128, 128, 128]

        up_1 = self.up1(d2)                       # => [N, 64, 256, 256]
        cat_1 = torch.cat((up_1, e1), dim=1)      # => [N, 128, 256, 256]
        d1 = self.dec1(cat_1)                     # => [N, 64, 256, 256]

        # ----------------- HEAD -----------------
        out = self.conv_out(d1)                   # => [N, out_channels, 256, 256]
        return out

    @staticmethod
    def _block(in_channels, out_channels, name="block"):
        """
        A basic U-Net block: 2 convolutions with ReLU (or optionally BatchNorm, etc).
        For an 'amazing' network, you can add BatchNorm or dropout here if you like.
        """
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

Testing on a small subset:

In [ ]:
subset_indices = range(50)
train_dataset_small = Subset(train_dataset, subset_indices)
train_loader_small = DataLoader(
    train_dataset_small,
    batch_size=4,
    shuffle=True,
    num_workers=2
)
val_dataset_small = Subset(val_dataset, subset_indices)
val_loader_small = DataLoader(
    val_dataset_small,
    batch_size=4,
    shuffle=False,
    num_workers=2
)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNet(in_channels=2, out_channels=2).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-6)

# Suppose your patches are indeed (2,256,256)
dummy_input = torch.randn(1, 2, 256, 256).to(device)
out = model(dummy_input)
print("Output shape:", out.shape)
# Should be [1, 2, 256, 256] if everything matches

In [ ]:
print("Using device:", device)

In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

#Noise2Void (N2V) Training Run:


#Noise2Void Training Function:

This function implements a self-supervised Noise2Void denoising pipeline with several key features:

    1.  Random Pixel Masking (Noise2Void)
        - Each batch, a fraction of pixels is hidden (set to 0).
        - The model learns to predict these masked pixels from surrounding context,
          removing noise with no clean reference required.

    2.  Mixed Precision Acceleration
        - Utilizes PyTorch AMP (autocast + GradScaler) for faster training
          and reduced GPU memory usage.

    3.  Training & Validation
        - After each training epoch, a validation pass (using the same masking approach)
          checks the model’s generalization.
        - Helps detect overfitting or potential numeric issues.

    4.  Checkpoint Loading & Saving
        - Optionally resumes from a previous checkpoint (if provided),
          restoring the model and optimizer states.
        - Saves a new checkpoint at the end of every epoch
          (e.g., 'checkpoint_epochN.pth').

    5.  Partial Progress Printing
        - Prints training loss “so far” every 'print_interval' mini-batches
          to track improvement within each epoch.
        - Also prints final train and validation loss at epoch’s end.

    6.  Early Stopping
        - Monitors validation loss each epoch.
        - If no improvement is seen for a specified patience, training stops early
          to avoid overfitting.

    7.  Skipping Problematic Batches
        - If inputs, outputs, or the computed loss are NaN/Inf, the code
          skips that batch to prevent updating with invalid data.

    8.  Gradient Clipping
        - After backward but before optimizer.step(), gradients are clipped
          (e.g., via clip_grad_norm_) to prevent exploding updates
          that can cause NaNs or instability.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils as utils  # for gradient clipping
import numpy as np
from torch import amp
import math
import os

def noise2void_train_and_validate(
    model,
    train_loader,
    val_loader,
    device='cuda',
    # Hyperparams
    num_epochs=30,
    mask_ratio=0.1,
    print_interval=50,
    resume_checkpoint=None,
    patience=5,
    # Optimizer config
    lr=5e-6,  # even lower LR than 1e-5
    betas=(0.9, 0.95),
    eps=1e-7,
    # Clipping config
    grad_clip_max_norm=0.05,  # stronger clipping
    # Debug toggles
    debug=False
):
    """
    Noise2Void training + validation with comprehensive numeric safeguards:

    1. Uses random pixel masking (Noise2Void).
    2. Mixed precision (autocast + GradScaler).
    3. Checkpoint loading & saving.
    4. Early stopping if val loss doesn't improve for 'patience' epochs.
    5. NaN/Inf checks on inputs, outputs, and loss (skips those batches).
    6. Strong gradient clipping (max_norm can be tightened).
    7. Debug prints for param min/max each epoch to detect weight explosions.

    Args:
        model (nn.Module): Your U-Net (or other) model in log or linear domain.
        train_loader (DataLoader): Loader for training patches.
        val_loader (DataLoader): Loader for validation patches.
        device (str): 'cuda' or 'cpu'.
        num_epochs (int): Max epochs to train.
        mask_ratio (float): fraction of pixels masked each iteration.
        print_interval (int): prints partial train loss every N mini-batches.
        resume_checkpoint (str): path to a saved checkpoint to resume from.
        patience (int): early stopping patience for val loss improvement.
        lr (float): learning rate for Adam.
        betas (tuple): Adam betas for momentum/variance.
        eps (float): Adam epsilon for numeric stability.
        grad_clip_max_norm (float): how strictly to clip gradients each batch.
        debug (bool): if True, prints param range each epoch to diagnose weight blow-ups.
    """
    # 1) Create the optimizer with the specified hyperparams
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        betas=betas,
        eps=eps
    )

    # 2) Mixed precision gradient scaler
    scaler = amp.GradScaler()

    start_epoch = 1
    best_val_loss = float('inf')
    no_improve_count = 0
    best_epoch = 1

    # 3) Resume if a checkpoint is provided
    if resume_checkpoint is not None and os.path.isfile(resume_checkpoint):
        ckpt = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from checkpoint '{resume_checkpoint}' at epoch {ckpt['epoch']}.")

    # 4) Main training loop
    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        running_loss = 0.0
        total_train_batches = len(train_loader)

        for batch_idx, (inputs, _) in enumerate(train_loader, start=1):
            # Check for NaN/inf in inputs
            if inputs.isnan().any() or inputs.isinf().any():
                print(f"[Train] Skipping batch {batch_idx} (NaN/inf in inputs).")
                continue

            inputs = inputs.to(device)
            targets = inputs.clone()

            B, C, H, W = inputs.shape
            # Random mask
            mask = (torch.rand(B, 1, H, W, device=device) < mask_ratio).float()
            mask = mask.repeat(1, C, 1, 1)
            sum_mask = mask.sum()
            if sum_mask < 1e-5:
                # skip if almost no masked pixels
                continue

            masked_inputs = inputs * (1 - mask)

            optimizer.zero_grad()
            with amp.autocast(device_type='cuda'):
                outputs = model(masked_inputs)

                # Check outputs
                if outputs.isnan().any() or outputs.isinf().any():
                    print(f"[Train] Skipping batch {batch_idx} (NaN/inf in outputs).")
                    continue

                # Compute MSE on masked pixels
                loss_map = criterion_no_reduction(outputs, targets)
                masked_loss_map = loss_map * mask
                loss = masked_loss_map.sum() / sum_mask

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"[Train] Skipping batch {batch_idx} (loss is NaN/inf).")
                    continue

            # Backward with gradient clipping
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_max_norm)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            # partial progress print
            if (batch_idx % print_interval) == 0:
                avg_so_far = running_loss / batch_idx
                print(f"[Train] Epoch [{epoch}/{num_epochs}], "
                      f"Batch [{batch_idx}/{total_train_batches}] → "
                      f"AvgLossSoFar: {avg_so_far:.4f}")

        train_epoch_loss = running_loss / total_train_batches
        print(f"--- Epoch [{epoch}/{num_epochs}] done - Noise2Void Train Loss: {train_epoch_loss:.4f}")

        # (Optional) Debug: check param ranges after each epoch
        if debug:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        pmin = param.data.min().item()
                        pmax = param.data.max().item()
                        print(f"Param [{name}] range: {pmin:.4e} to {pmax:.4e}")

        # ------------------- Validation -------------------
        model.eval()
        val_running_loss = 0.0
        total_val_batches = len(val_loader)

        with torch.no_grad():
            for val_batch_idx, (val_inp, _) in enumerate(val_loader, start=1):
                if val_inp.isnan().any() or val_inp.isinf().any():
                    print(f"[Val] Skipping batch {val_batch_idx} (NaN/inf in inputs).")
                    continue

                val_inp = val_inp.to(device)
                val_tgt = val_inp.clone()

                Bv, Cv, Hv, Wv = val_inp.shape
                val_mask = (torch.rand(Bv, 1, Hv, Wv, device=device) < mask_ratio).float()
                val_mask = val_mask.repeat(1, Cv, 1, 1)
                sum_val_mask = val_mask.sum()
                if sum_val_mask < 1e-5:
                    continue

                with amp.autocast(device_type='cuda'):
                    val_out = model(val_inp * (1 - val_mask))

                    if val_out.isnan().any() or val_out.isinf().any():
                        print(f"[Val] Skipping batch {val_batch_idx} (NaN/inf in outputs).")
                        continue

                    val_loss_map = criterion_no_reduction(val_out, val_tgt)
                    masked_val_loss_map = val_loss_map * val_mask
                    val_loss = masked_val_loss_map.sum() / sum_val_mask

                    if torch.isnan(val_loss) or torch.isinf(val_loss):
                        print(f"[Val] Skipping batch {val_batch_idx} (val_loss is NaN/inf).")
                        continue

                val_running_loss += val_loss.item()

        val_epoch_loss = val_running_loss / total_val_batches
        print(f"Validation Loss after Epoch [{epoch}/{num_epochs}]: {val_epoch_loss:.4f}")

        # ------------------- Early Stopping -------------------
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            no_improve_count = 0
            best_epoch = epoch
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"No improvement for {patience} consecutive epochs. Early stopping.")
                break

        # ------------------- Save Checkpoint -------------------
        ckpt_path = f"checkpoint_epoch{epoch}.pth"
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
        }, ckpt_path)
        print(f"Checkpoint saved: {ckpt_path}")

    print(f"Training finished. Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils as utils  # for gradient clipping
import numpy as np
import math
import os

def noise2void_train_and_validate_no_amp_debug(
    model,
    train_loader,
    val_loader,
    criterion_no_reduction,
    device='cuda',
    num_epochs=30,
    mask_ratio=0.1,
    print_interval=50,
    resume_checkpoint=None,
    patience=5,
    lr=1e-5,
    betas=(0.9, 0.95),
    eps=1e-7,
    grad_clip_max_norm=0.1
):
    """
    Noise2Void training + validation WITHOUT AMP, using custom Adam hyperparams,
    printing stats each batch for debugging, and skipping NaN/Inf batches.

    Args:
        model (nn.Module): U-Net or similar network.
        train_loader (DataLoader): Training dataset loader.
        val_loader (DataLoader): Validation dataset loader.
        criterion_no_reduction (nn.MSELoss with reduction='none'):
            Pixelwise MSE for Noise2Void.
        device (str): 'cuda' or 'cpu'.
        num_epochs (int): Max training epochs.
        mask_ratio (float): fraction of pixels to mask per batch.
        print_interval (int): prints debug info every N mini-batches.
        resume_checkpoint (str): path to a checkpoint to resume from.
        patience (int): early stopping patience for val loss improvement.
        lr (float): learning rate for Adam.
        betas (tuple): (beta1, beta2) for Adam momentum.
        eps (float): Adam epsilon for numeric stability.
        grad_clip_max_norm (float): gradient clipping max norm (L2).
    """

    # Create the optimizer with custom Adam hyperparams
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        betas=betas,
        eps=eps
    )

    start_epoch = 1
    best_val_loss = float('inf')
    no_improve_count = 0
    best_epoch = 1

    # Optionally resume from a checkpoint
    if resume_checkpoint is not None and os.path.isfile(resume_checkpoint):
        ckpt = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from checkpoint '{resume_checkpoint}' at epoch {ckpt['epoch']}.")

    model.to(device)

    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        running_loss = 0.0
        total_train_batches = len(train_loader)

        for batch_idx, (inputs, _) in enumerate(train_loader, start=1):
            # Check for NaN/inf in inputs
            if inputs.isnan().any() or inputs.isinf().any():
                print(f"[Train] E{epoch} B{batch_idx}: Skipping (NaN/inf in inputs).")
                continue

            inputs = inputs.to(device)
            targets = inputs.clone()  # Noise2Void => same as inputs

            B, C, H, W = inputs.shape
            mask = (torch.rand(B, 1, H, W, device=device) < mask_ratio).float()
            mask = mask.repeat(1, C, 1, 1)
            sum_mask = mask.sum()
            if sum_mask < 1e-5:
                # skip if too few masked pixels
                continue

            masked_inputs = inputs * (1 - mask)

            # Stats for debugging
            in_min = inputs.min().item()
            in_max = inputs.max().item()
            masked_in_min = masked_inputs.min().item()
            masked_in_max = masked_inputs.max().item()

            optimizer.zero_grad()
            outputs = model(masked_inputs)

            if outputs.isnan().any() or outputs.isinf().any():
                print(f"[Train] E{epoch} B{batch_idx}: Skipping (NaN/inf in outputs).")
                print(f"    in[{in_min:.4f},{in_max:.4f}] masked[{masked_in_min:.4f},{masked_in_max:.4f}]")
                continue

            out_min = outputs.min().item()
            out_max = outputs.max().item()

            # Noise2Void loss
            loss_map = criterion_no_reduction(outputs, targets)
            masked_loss_map = loss_map * mask
            loss = masked_loss_map.sum() / sum_mask

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[Train] E{epoch} B{batch_idx}: Skipping (NaN/inf loss).")
                print(f"    in[{in_min:.4f},{in_max:.4f}] masked[{masked_in_min:.4f},{masked_in_max:.4f}] "
                      f"out[{out_min:.4f},{out_max:.4f}]")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_max_norm)
            optimizer.step()

            running_loss += loss.item()

            # Print partial debug info
            if (batch_idx % print_interval) == 0:
                avg_so_far = running_loss / batch_idx
                print(f"[Train] E{epoch} B{batch_idx}/{total_train_batches} "
                      f"in[{in_min:.4f},{in_max:.4f}] mIn[{masked_in_min:.4f},{masked_in_max:.4f}] "
                      f"out[{out_min:.4f},{out_max:.4f}] loss={loss.item():.4f} AvgSoFar={avg_so_far:.4f}")

        # End of epoch
        train_epoch_loss = running_loss / total_train_batches
        print(f"--- Epoch [{epoch}/{num_epochs}] done - Noise2Void Train Loss: {train_epoch_loss:.4f}")

        # ------------------- Validation -------------------
        model.eval()
        val_running_loss = 0.0
        total_val_batches = len(val_loader)

        with torch.no_grad():
            for val_batch_idx, (val_inp, _) in enumerate(val_loader, start=1):
                if val_inp.isnan().any() or val_inp.isinf().any():
                    print(f"[Val] E{epoch} B{val_batch_idx}: Skipping (NaN/inf in inputs).")
                    continue

                val_inp = val_inp.to(device)
                val_tgt = val_inp.clone()

                Bv, Cv, Hv, Wv = val_inp.shape
                val_mask = (torch.rand(Bv, 1, Hv, Wv, device=device) < mask_ratio).float()
                val_mask = val_mask.repeat(1, Cv, 1, 1)
                sum_val_mask = val_mask.sum()
                if sum_val_mask < 1e-5:
                    continue

                val_out = model(val_inp * (1 - val_mask))
                if val_out.isnan().any() or val_out.isinf().any():
                    print(f"[Val] E{epoch} B{val_batch_idx}: Skipping (NaN/inf in outputs).")
                    continue

                val_loss_map = criterion_no_reduction(val_out, val_tgt)
                masked_val_loss_map = val_loss_map * val_mask
                val_loss = masked_val_loss_map.sum() / sum_val_mask

                if torch.isnan(val_loss) or torch.isinf(val_loss):
                    print(f"[Val] E{epoch} B{val_batch_idx}: Skipping (NaN/inf val_loss).")
                    continue

                val_running_loss += val_loss.item()

        val_epoch_loss = val_running_loss / total_val_batches
        print(f"Validation Loss after Epoch [{epoch}/{num_epochs}]: {val_epoch_loss:.4f}")

        # Early Stopping
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            no_improve_count = 0
            best_epoch = epoch
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"No improvement for {patience} consecutive epochs. Early stopping.")
                break

        # Save checkpoint
        ckpt_path = f"checkpoint_epoch{epoch}.pth"
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
        }, ckpt_path)
        print(f"Checkpoint saved: {ckpt_path}")

    print(f"Training finished. Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}.")

In [ ]:
#Load a Checkpoint
!cp "/content/drive/MyDrive/sentineldata/checkpoints/checkpoint_epoch16.pth" "/content/checkpoint_epoch16.pth"

In [ ]:
val_iter = iter(val_loader)
inputs, _ = next(val_iter)
print("Inputs min/max:", inputs.min(), inputs.max())

In [ ]:
criterion_no_reduction = nn.MSELoss(reduction='none')


noise2void_train_and_validate_no_amp_debug(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_no_reduction=criterion_no_reduction,
    device='cuda',
    num_epochs=30,
    mask_ratio=0.1,
    print_interval=50,
    resume_checkpoint='checkpoint_epoch16.pth',
    patience=5,
    lr=5e-6,                # super-low LR
    betas=(0.9, 0.95),
    eps=1e-7,
    grad_clip_max_norm=0.05,

)

In [ ]:
!mkdir -p "/content/drive/MyDrive/sentineldata/checkpoints/"

In [ ]:
# Copy the final (or best) checkpoint to Drive
!cp "/content/checkpoint_epoch24.pth" "/content/drive/MyDrive/sentineldata/checkpoints/"

#Now we can visualise our models performance


In [ ]:
##############################################
# 1) No-Reference Metrics
##############################################
def compute_enl(img_lin):
    """
    Equivalent Number of Looks (ENL) in linear domain:
      ENL = (mean^2) / (std^2)
    """
    mu = np.mean(img_lin)
    sigma = np.std(img_lin)
    return (mu*mu) / (sigma*sigma + 1e-9)

def compute_cv(img_lin):
    """
    Coefficient of Variation (CV) = std / mean (linear domain).
    """
    mu = np.mean(img_lin)
    sigma = np.std(img_lin)
    return sigma / (mu + 1e-9)


##############################################
# 2) Patch-Based Inference & Stitching
##############################################
def infer_and_stitch(model, big_image_log, patch_size=1024, stride=1024, device='cuda'):
    """
    Splits (C,H,W) log-domain data into patches, runs model, and stitches results.
    Overlapping areas get averaged.
    Returns a stitched log-domain array of shape (C,H,W).
    (We assume your model works in log space.)
    """
    model.eval()
    C, H, W = big_image_log.shape
    big_image_t = torch.from_numpy(big_image_log).unsqueeze(0).float().to(device)  # [1,C,H,W]

    stitched_vals = torch.zeros((C, H, W), device=device)
    stitched_count = torch.zeros((C, H, W), device=device)

    with torch.no_grad():
        row = 0
        while row + patch_size <= H:
            col = 0
            while col + patch_size <= W:
                patch_t = big_image_t[:, :, row:row+patch_size, col:col+patch_size]
                out_patch = model(patch_t).squeeze(0)  # shape (C, patch_size, patch_size)

                stitched_vals[:, row:row+patch_size, col:col+patch_size] += out_patch
                stitched_count[:, row:row+patch_size, col:col+patch_size] += 1.0

                col += stride
            row += stride

    stitched_count = torch.clamp_min(stitched_count, 1e-9)
    stitched_vals /= stitched_count
    return stitched_vals.cpu().numpy()  # log-domain output


##############################################
# 3) Visualization Helpers
##############################################
def show_three_subplots(orig, den, title_prefix="Channel0"):
    """
    Plots [Original, Denoised, Difference] for a single channel (both in log).
    Uses a shared percentile-based stretch for orig/den, then a diverging colormap for diff.
    """
    diff = den - orig

    combined = np.concatenate([orig.flatten(), den.flatten()])
    vmin = np.percentile(combined, 2)
    vmax = np.percentile(combined, 98)

    diff_min = np.percentile(diff, 2)
    diff_max = np.percentile(diff, 98)

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    axs[0].imshow(orig, cmap='gray', vmin=vmin, vmax=vmax)
    axs[0].set_title(f"{title_prefix} Original (Log)")
    axs[0].axis('off')

    axs[1].imshow(den, cmap='gray', vmin=vmin, vmax=vmax)
    axs[1].set_title(f"{title_prefix} Denoised (Log)")
    axs[1].axis('off')

    axs[2].imshow(diff, cmap='bwr', vmin=diff_min, vmax=diff_max)
    axs[2].set_title(f"{title_prefix} Difference (Log)")
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()


def show_color_composite(orig_vv, orig_vh, den_vv, den_vh):
    """
    Creates color composites (R=VV, G=VH, B=(VV+VH)/2) for both original and denoised,
    plus a difference composite in log space.
    """
    # Construct color images in log domain: R=VV, G=VH, B=0.5*(VV+VH).
    orig_color = np.stack([orig_vv, orig_vh, 0.5*(orig_vv + orig_vh)], axis=-1)
    den_color  = np.stack([den_vv,  den_vh,  0.5*(den_vv + den_vh)],  axis=-1)
    diff_color = den_color - orig_color

    combined_oc = np.concatenate([orig_color.flatten(), den_color.flatten()])
    vmin = np.percentile(combined_oc, 2)
    vmax = np.percentile(combined_oc, 98)

    diff_min = np.percentile(diff_color, 2)
    diff_max = np.percentile(diff_color, 98)

    fig, axs = plt.subplots(1, 3, figsize=(15,5))

    # Original
    orig_norm = (orig_color - vmin) / (vmax - vmin + 1e-9)
    orig_norm = np.clip(orig_norm, 0, 1)
    axs[0].imshow(orig_norm)
    axs[0].set_title("Original Color Composite (Log)")
    axs[0].axis('off')

    # Denoised
    den_norm = (den_color - vmin) / (vmax - vmin + 1e-9)
    den_norm = np.clip(den_norm, 0, 1)
    axs[1].imshow(den_norm)
    axs[1].set_title("Denoised Color Composite (Log)")
    axs[1].axis('off')

    # Difference
    diff_norm = (diff_color - diff_min) / (diff_max - diff_min + 1e-9)
    diff_norm = np.clip(diff_norm, 0, 1)
    axs[2].imshow(diff_norm)
    axs[2].set_title("Difference Color Composite (Log)")
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()


##############################################
# 4) The Main Folder-Loop + Metrics
##############################################
def visualize_all_folders_only_checkpoint24(
    root_dir,
    checkpoint_path,
    device='cuda',
    patch_size=1024,
    stride=1024
):
    """
    1) Loops over each subfolder in root_dir, searching for VV & VH .tiff files.
    2) For each folder, loads the data in linear domain => logs it.
    3) Uses patch-based inference with the given model checkpoint.
    4) Plots per-channel subplots + color composite.
    5) Prints ENL & CV in linear domain for Original vs. Denoised, both channels.
    """
    # Access your existing UNet
    # (We do NOT define the class here; you must define or import it above)
    model = UNet(in_channels=2, out_channels=2).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # Find VV/VH tiffs
        vv_path, vh_path = None, None
        for fname in os.listdir(folder_path):
            f_lower = fname.lower()
            if f_lower.endswith(".tiff") and "vv" in f_lower:
                vv_path = os.path.join(folder_path, fname)
            elif f_lower.endswith(".tiff") and "vh" in f_lower:
                vh_path = os.path.join(folder_path, fname)

        if vv_path is None or vh_path is None:
            print(f"Skipping {folder_name}: no vv/vh tiffs found.")
            continue

        # Read data in linear domain => shape (2,H,W)
        with rasterio.open(vv_path) as ds_vv:
            arr_vv = ds_vv.read(1).astype(np.float32)
        with rasterio.open(vh_path) as ds_vh:
            arr_vh = ds_vh.read(1).astype(np.float32)

        big_image_lin = np.stack([arr_vv, arr_vh], axis=0)

        # Convert to log => shape (2,H,W)
        big_image_log = np.log1p(big_image_lin)

        # Patch-based denoising in log
        denoised_log = infer_and_stitch(
            model=model,
            big_image_log=big_image_log,
            patch_size=patch_size,
            stride=stride,
            device=device
        )

        print(f"\n=== Folder: {folder_name} ===")
        print("Original shape:", big_image_lin.shape)

        # Show subplots: channel 0 => VV, channel 1 => VH
        show_three_subplots(orig=big_image_log[0], den=denoised_log[0], title_prefix="VV")
        show_three_subplots(orig=big_image_log[1], den=denoised_log[1], title_prefix="VH")

        # Color composite in log domain
        show_color_composite(
            orig_vv=big_image_log[0],
            orig_vh=big_image_log[1],
            den_vv=denoised_log[0],
            den_vh=denoised_log[1]
        )

        # --- Compute ENL/CV in linear domain
        # Original => big_image_lin
        # Denoised => np.expm1(denoised_log)
        den_lin = np.expm1(denoised_log)  # shape (2,H,W) in linear domain

        # Original VV
        enl_orig_vv = compute_enl(big_image_lin[0])
        cv_orig_vv  = compute_cv(big_image_lin[0])
        # Denoised VV
        enl_den_vv = compute_enl(den_lin[0])
        cv_den_vv  = compute_cv(den_lin[0])

        # Original VH
        enl_orig_vh = compute_enl(big_image_lin[1])
        cv_orig_vh  = compute_cv(big_image_lin[1])
        # Denoised VH
        enl_den_vh = compute_enl(den_lin[1])
        cv_den_vh  = compute_cv(den_lin[1])

        print("----- ENL / CV Metrics (Linear Domain) -----")
        print(f"VV Original:   ENL={enl_orig_vv:.3f}, CV={cv_orig_vv:.3f}")
        print(f"VV Denoised:   ENL={enl_den_vv:.3f}, CV={cv_den_vv:.3f}")
        print(f"VH Original:   ENL={enl_orig_vh:.3f}, CV={cv_orig_vh:.3f}")
        print(f"VH Denoised:   ENL={enl_den_vh:.3f}, CV={cv_den_vh:.3f}")
        print("-------------------------------------------")

        # Cleanup
        del arr_vv, arr_vh, big_image_lin, big_image_log, denoised_log, den_lin
        torch.cuda.empty_cache()


##############################################
# 5) Final Invocation
##############################################
# Just call this at the end of your script (or in a cell):
# (Of course, you must define your "UNet(in_channels=2, out_channels=2)" above.)

visualize_all_folders_only_checkpoint24(
    root_dir="/content/drive/MyDrive/sentineldata/tiff-files/",
    checkpoint_path="/content/drive/MyDrive/sentineldata/checkpoints/checkpoint_epoch24.pth",
    device='cuda',       # or 'cpu'
    patch_size=1024,
    stride=1024
)

In [ ]:
def process_vv_vh_subregion_with_metrics(
    vv_path,
    vh_path,
    row,
    col,
    sub_size,
    model,
    device='cuda'
):
    """
    1) Reads a subregion (row,col,sub_size) from vv_path & vh_path in linear domain.
    2) Applies classical filters (Lee/Frost/Kuan) to VH in log domain,
    3) Does a double-pass N2V for 2-ch input (VV+VH) in log domain,
    4) Plots a 2×6 figure for VH [Original,Lee,Frost,Kuan,Pass1,Pass2 + differences],
    5) Prints ENL/CV for Original(VH), Lee, Frost, Kuan, N2V pass1, pass2.
    """
    # 1) Read full images
    with rasterio.open(vv_path) as ds_vv:
        vv_full = ds_vv.read(1).astype(np.float32)
    with rasterio.open(vh_path) as ds_vh:
        vh_full = ds_vh.read(1).astype(np.float32)

    # 2) Extract subregion
    sub_vv_lin = vv_full[row:row+sub_size, col:col+sub_size]
    sub_vh_lin = vh_full[row:row+sub_size, col:col+sub_size]
    sub_2ch_lin = np.stack([sub_vv_lin, sub_vh_lin], axis=0)

    # "Original" for VH
    orig_vh_lin = sub_vh_lin

    # 3) Classical filters on VH in log domain
    lee_vh   = apply_filter_in_log_domain(orig_vh_lin, lee_filter)
    frost_vh = apply_filter_in_log_domain(orig_vh_lin, frost_filter)
    kuan_vh  = apply_filter_in_log_domain(orig_vh_lin, kuan_filter)

    # 4) Double-pass N2V
    pass1_2ch_lin, pass2_2ch_lin = double_pass_n2v_unet(sub_2ch_lin, model, device=device)
    pass1_vh_lin = pass1_2ch_lin[1]
    pass2_vh_lin = pass2_2ch_lin[1]

    # Differences
    lee_diff   = lee_vh   - orig_vh_lin
    frost_diff = frost_vh - orig_vh_lin
    kuan_diff  = kuan_vh  - orig_vh_lin
    pass1_diff = pass1_vh_lin - orig_vh_lin
    pass2_diff = pass2_vh_lin - orig_vh_lin

    # 5) Plot 2×6 figure (VH)
    fig, axes = plt.subplots(2, 6, figsize=(22,7))
    fig.suptitle(
        f"Folder Subregion row={row}, col={col}, size={sub_size}\n"
        f"VV={os.path.basename(vv_path)}, VH={os.path.basename(vh_path)}",
        fontsize=14
    )

    # top row => Original, Lee, Frost, Kuan, pass1, pass2
    show_clipped(axes[0,0], orig_vh_lin,   title="Original(VH)")
    show_clipped(axes[0,1], lee_vh,        title="Lee(VH)")
    show_clipped(axes[0,2], frost_vh,      title="Frost(VH)")
    show_clipped(axes[0,3], kuan_vh,       title="Kuan(VH)")
    show_clipped(axes[0,4], pass1_vh_lin,  title="N2V pass1(VH)")
    show_clipped(axes[0,5], pass2_vh_lin,  title="N2V pass2(VH)")

    # bottom row => differences
    axes[1,0].axis('off')
    axes[1,0].text(0.02, 0.5, "Diff vs. Original(VH)",
                   fontsize=12, ha='left', va='center', transform=axes[1,0].transAxes)
    show_diff(axes[1,1], lee_diff,   title="Lee Diff")
    show_diff(axes[1,2], frost_diff, title="Frost Diff")
    show_diff(axes[1,3], kuan_diff,  title="Kuan Diff")
    show_diff(axes[1,4], pass1_diff, title="N2V1 Diff")
    show_diff(axes[1,5], pass2_diff, title="N2V2 Diff")

    plt.tight_layout()
    plt.show()

    # 6) Print ENL/CV for each method (VH channel in linear domain)
    enl_orig = compute_enl(orig_vh_lin)
    cv_orig  = compute_cv(orig_vh_lin)
    enl_lee  = compute_enl(lee_vh)
    cv_lee   = compute_cv(lee_vh)
    enl_fr   = compute_enl(frost_vh)
    cv_fr    = compute_cv(frost_vh)
    enl_kn   = compute_enl(kuan_vh)
    cv_kn    = compute_cv(kuan_vh)
    enl_p1   = compute_enl(pass1_vh_lin)
    cv_p1    = compute_cv(pass1_vh_lin)
    enl_p2   = compute_enl(pass2_vh_lin)
    cv_p2    = compute_cv(pass2_vh_lin)

    print(f"\nMetrics for row={row}, col={col}, size={sub_size} (VH):")
    print(f"  Original: ENL={enl_orig:.3f}, CV={cv_orig:.3f}")
    print(f"  Lee:      ENL={enl_lee:.3f}, CV={cv_lee:.3f}")
    print(f"  Frost:    ENL={enl_fr:.3f}, CV={cv_fr:.3f}")
    print(f"  Kuan:     ENL={enl_kn:.3f}, CV={cv_kn:.3f}")
    print(f"  N2V p1:   ENL={enl_p1:.3f}, CV={cv_p1:.3f}")
    print(f"  N2V p2:   ENL={enl_p2:.3f}, CV={cv_p2:.3f}")
    print("-------------------------------------------")


def visualize_folders_multiple_subregions(
    root_dir,
    checkpoint_path,
    model,
    device='cuda',
    sub_size=512,
    N_RANDOM=2
):
    """
    Loops through each subfolder in root_dir, looking for .tiff files
    named with "vv" / "vh". For each folder, picks multiple random subregions
    (or 1 fixed subregion) to do the same subregion-based approach:
      1) read VV & VH (linear domain),
      2) classical filters on VH in log,
      3) double-pass N2V in log,
      4) plot 2×6 figure,
      5) print ENL/CV for VH.
    """
    # load checkpoint into model
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    model.to(device)

    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # find vv & vh
        vv_path, vh_path = None, None
        for fname in os.listdir(folder_path):
            lf = fname.lower()
            if lf.endswith(".tiff") and "vv" in lf:
                vv_path = os.path.join(folder_path, fname)
            elif lf.endswith(".tiff") and "vh" in lf:
                vh_path = os.path.join(folder_path, fname)

        if vv_path is None or vh_path is None:
            print(f"Skipping {folder_name}: no vv/vh tiffs found.")
            continue

        # read full arrays
        with rasterio.open(vv_path) as ds_vv:
            vv_data = ds_vv.read(1).astype(np.float32)
            H, W = vv_data.shape
        with rasterio.open(vh_path) as ds_vh:
            vh_data = ds_vh.read(1).astype(np.float32)
            H2, W2 = vh_data.shape
        if (H != H2) or (W != W2):
            print(f"Folder {folder_name}: mismatch shape for VV vs VH. Skipping.")
            continue

        print(f"\n=== Folder: {folder_name} === shape=({H},{W})")

        # 1) One fixed subregion
        row_fixed, col_fixed = 8232, 15410
        if row_fixed + sub_size <= H and col_fixed + sub_size <= W:
            print(f"[Fixed] row={row_fixed}, col={col_fixed}, sub_size={sub_size}")
            process_vv_vh_subregion_with_metrics(
                vv_path=vv_path,
                vh_path=vh_path,
                row=row_fixed,
                col=col_fixed,
                sub_size=sub_size,
                model=model,
                device=device
            )
        else:
            print("[Fixed] subregion out of bounds for this folder, skipping fixed subregion.")

        # 2) multiple random subregions
        max_row = H - sub_size
        max_col = W - sub_size
        if max_row < 0 or max_col < 0:
            print(f"Image too small for sub_size={sub_size}. Skipping random subregions.")
            continue

        for i in range(N_RANDOM):
            rand_row = random.randint(0, max_row)
            rand_col = random.randint(0, max_col)
            print(f"[Random] row={rand_row}, col={rand_col}, size={sub_size}")
            process_vv_vh_subregion_with_metrics(
                vv_path=vv_path,
                vh_path=vh_path,
                row=rand_row,
                col=rand_col,
                sub_size=sub_size,
                model=model,
                device=device
            )

In [ ]:
# Above: define or import your classical filters & apply_filter_in_log_domain,
# define or import compute_enl, compute_cv,
# define or import your double_pass_n2v_unet, etc.
# define or import your "class UNet(in_channels=2,out_channels=2)"

root_dir = "/content/drive/MyDrive/sentineldata/tiff-files/"
checkpoint_path = "/content/drive/MyDrive/sentineldata/checkpoints/checkpoint_epoch24.pth"

# Then call:
visualize_folders_multiple_subregions(
    root_dir=root_dir,
    checkpoint_path=checkpoint_path,
    model=UNet(in_channels=2, out_channels=2),  # or your existing instance
    device='cuda',
    sub_size=512,
    N_RANDOM=2
)